## Environment Setup
Majority of the setup was taken from Professor's code on how to fix the Java issue on Google Colab. This may be a temporary solution until it is fixed on Colab, but in the meantime run this setup every time you start a kernel.

In [1]:
!sudo apt-get install -y openjdk-17-jdk

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  ca-certificates-java fonts-dejavu-core fonts-dejavu-extra java-common
  libatk-wrapper-java libatk-wrapper-java-jni libpcsclite1 libxt-dev libxtst6
  libxxf86dga1 openjdk-17-jdk-headless openjdk-17-jre openjdk-17-jre-headless
  x11-utils
Suggested packages:
  default-jre pcscd libxt-doc openjdk-17-demo openjdk-17-source visualvm
  libnss-mdns fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  ca-certificates-java fonts-dejavu-core fonts-dejavu-extra java-common
  libatk-wrapper-java libatk-wrapper-java-jni libpcsclite1 libxt-dev libxtst6
  libxxf86dga1 openjdk-17-jdk openjdk-17-jdk-headless openjdk-17-jre
  openjdk-17-jre-headless x11-utils
0 upgraded, 15 newly installed, 0 to remove and 41 not upgraded.
Need to get 125 MB of archives.


These bottom two code blocks are just for verifying that the packages are installed, not mandatory to run every time.

In [2]:
import os, subprocess
print("JAVA_HOME =", os.environ.get("JAVA_HOME"))
!echo $JAVA_HOME && $JAVA_HOME/bin/java -version

JAVA_HOME = None

openjdk version "17.0.16" 2025-07-15
OpenJDK Runtime Environment (build 17.0.16+8-Ubuntu-0ubuntu122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.16+8-Ubuntu-0ubuntu122.04.1, mixed mode, sharing)


In [3]:
!java -version || echo "No Java"
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

openjdk version "17.0.16" 2025-07-15
OpenJDK Runtime Environment (build 17.0.16+8-Ubuntu-0ubuntu122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.16+8-Ubuntu-0ubuntu122.04.1, mixed mode, sharing)
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35


## Download and Setup Files

In [4]:
# Line fetched from Kaggle's cURL option, modified to fit Colab's file system
!curl -L -o ./ultimate-spotify-tracks-db.zip \
  https://www.kaggle.com/api/v1/datasets/download/zaheenhamidani/ultimate-spotify-tracks-db

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 15.4M  100 15.4M    0     0  40.3M      0 --:--:-- --:--:-- --:--:-- 40.3M


In [5]:
!unzip ultimate-spotify-tracks-db.zip
!ls

Archive:  ultimate-spotify-tracks-db.zip
  inflating: SpotifyFeatures.csv     
sample_data  SpotifyFeatures.csv  ultimate-spotify-tracks-db.zip


## Main Code

In [6]:
import os
from pyspark.sql import SparkSession

BASE = os.path.abspath(".")
WAREHOUSE_URI=f"file:{BASE}/warehouse"

spark = (SparkSession.builder
         .appName("CS131 - WeTheBestMusic")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.warehouse.dir", WAREHOUSE_URI)
         .getOrCreate())

spark

**Data preparation (PySpark)**

In [7]:

from pyspark.sql import functions as F
df_raw = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv("/content/SpotifyFeatures.csv"))

print("Raw rows:", df_raw.count())
df_raw.printSchema()
df_raw.show(5, truncate=False)

Raw rows: 232725
root
 |-- genre: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- track_id: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- acousticness: string (nullable = true)
 |-- danceability: string (nullable = true)
 |-- duration_ms: string (nullable = true)
 |-- energy: string (nullable = true)
 |-- instrumentalness: string (nullable = true)
 |-- key: string (nullable = true)
 |-- liveness: string (nullable = true)
 |-- loudness: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- speechiness: string (nullable = true)
 |-- tempo: string (nullable = true)
 |-- time_signature: string (nullable = true)
 |-- valence: string (nullable = true)

+-----+-----------------+--------------------------------+----------------------+----------+------------+------------+-----------+------+----------------+---+--------+--------+-----+-----------+-------+--------------+-------+
|genre|artist_

**Data Cleaning** and **Validation**


In [8]:
df = df_raw

# We treat track_id as the unique ID
PK_COL = "track_id"
POP_COL = "popularity"

# 1) Drop rows with missing track_id
df = df.filter(F.col(PK_COL).isNotNull())
print("Rows after step 1:", df.count())

# 2) Drop rows with invalid number of columns (caused by commas in track name)
df = df.filter(F.col(POP_COL).cast("int").isNotNull())
print("Rows after step 2:", df.count())

# 3) Fill missing artist/track names with "Unknown"
df = df.fillna({
    "artist_name": "Unknown",
    "track_name": "Unknown"
})
print("Rows after step 3:", df.count())

# 4) Remove exact duplicate rows
df = df.dropDuplicates(["track_id"])

print("Rows after cleaning:", df.count())
df.show(5, truncate=False)


Rows after step 1: 232725
Rows after step 2: 231704
Rows after step 3: 231704
Rows after cleaning: 175773
+------+-----------------+------------------------+----------------------+----------+------------+------------+-----------+------+----------------+---+--------+--------+-----+-----------+-------+--------------+-------+
|genre |artist_name      |track_name              |track_id              |popularity|acousticness|danceability|duration_ms|energy|instrumentalness|key|liveness|loudness|mode |speechiness|tempo  |time_signature|valence|
+------+-----------------+------------------------+----------------------+----------+------------+------------+-----------+------+----------------+---+--------+--------+-----+-----------+-------+--------------+-------+
|Anime |Capcom Sound Team|Zangief's Theme         |00021Wy6AyMbLP2tqij86e|13        |0.234       |0.617       |169173     |0.862 |0.976           |G  |0.141   |-12.855 |Major|0.0514     |129.578|4/4           |0.886  |
|Reggae|Mike Love 

In [9]:
# ----- Validate -----
print("Raw rows  :", df_raw.count())
print("Clean rows:", df.count())

# No null IDs
null_ids = df.filter(F.col("track_id").isNull()).count()
print("Null track_id:", null_ids)

# No duplicate IDs
dup_ids = df_raw.count() - df_raw.dropDuplicates(["track_id"]).count()
print("Duplicate track_id:", dup_ids)


# ----- Save cleaned data -----

df.write.mode("overwrite").parquet(f"{BASE}/spotify_cleaned_parquet")

import os, glob
temp_csv_path = f"{BASE}/spotify_cleaned_csv_temp"
final_csv_path = f"{BASE}/spotify_cleaned.csv"

(df.coalesce(1)
   .write
   .mode("overwrite")
   .option("header", True)
   .csv(temp_csv_path))

part_file = glob.glob(f"{temp_csv_path}/part-*.csv")[0]
os.rename(part_file, final_csv_path)
os.system(f"rm -r {temp_csv_path}")

Raw rows  : 232725
Clean rows: 175773
Null track_id: 0
Duplicate track_id: 56146


0

In [10]:
df = (spark.read.option("inferSchema", True).csv(f"{BASE}/spotify_cleaned.csv", header=True))

# Verify Schema Changes after Data Cleaning
df.printSchema()

root
 |-- genre: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- track_id: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- danceability: double (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- energy: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- key: string (nullable = true)
 |-- liveness: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: string (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: string (nullable = true)
 |-- valence: double (nullable = true)



## **Part 2: Non-Trivial Transformations (Simon)**
For this particular section, we decided to divide our data transformations into four different parts:
- Artist Statistics
  - Count of Tracks
  - Average Popularity of Artist
  - Average Duration of an Artist's Tracks (milliseconds)
- Genre Statistics
  - Count of Tracks
  - Average Popularity of Genre
  - Average BPM Genre Uses
- Artists with Respect to Time Signature
  - Count of Tracks
  - Average Popularity of Artist's Songs in Time Signature
  - Average Danceability
- Key and Mode Statistics
  - Count of Tracks
  - Average Popularity
  - Average Loudness
  - Average Energy

In [11]:
from pyspark.sql.functions import *
artist_statistics = df.groupBy("artist_name") \
  .agg(count("track_name").alias("number_of_tracks"),
       round(avg("popularity"), 3).alias("avg_popularity"),
       round(avg("duration_ms"), 3).alias("avg_length (ms)")) \
  .orderBy(desc("number_of_tracks"), desc("avg_popularity"))
artist_statistics.show()

genre_statistics = df.groupBy("genre") \
  .agg(count("track_name").alias("number_of_tracks"),
       round(avg("popularity"), 3).alias("avg_popularity"),
       round(avg("tempo"), 3).alias("avg_bpm")) \
  .orderBy(desc("number_of_tracks"), desc("avg_popularity"))
genre_statistics.show()

artists_by_signature = df.groupBy("artist_name","time_signature") \
  .agg(count("track_name").alias("number_of_tracks"),
       round(avg("popularity"), 3).alias("avg_popularity"),
       round(avg("danceability"), 3).alias("avg_danceability")) \
  .orderBy(desc("number_of_tracks"), desc("avg_popularity"))
artists_by_signature.show()

most_popular_signatures = df.groupBy("key", "mode") \
  .agg(count("track_name").alias("number_of_tracks"),
       round(avg("popularity"), 3).alias("avg_popularity"),
       round(avg("loudness"), 3).alias("avg_loudness"),
       round(avg("energy"), 3).alias("avg_energy")) \
  .orderBy("key", "mode")
most_popular_signatures.show()

+--------------------+----------------+--------------+---------------+
|         artist_name|number_of_tracks|avg_popularity|avg_length (ms)|
+--------------------+----------------+--------------+---------------+
|      Giuseppe Verdi|            1141|        13.363|     245712.916|
|Kimbo Children's ...|             971|         0.758|     129792.437|
|     Giacomo Puccini|             930|        13.848|      229661.36|
|Wolfgang Amadeus ...|             788|        22.025|     336658.726|
|       Nobuo Uematsu|             773|        22.916|     193407.159|
|         Juice Music|             684|         4.607|     129040.317|
|      Richard Wagner|             681|        12.722|     390001.436|
|        Randy Newman|             662|        14.526|      153535.66|
|       Georges Bizet|             627|        14.107|     231378.825|
|Johann Sebastian ...|             611|        26.448|     220585.969|
|Ludwig van Beethoven|             585|         24.28|     398405.769|
|     

## **Part 3: Data Analysis using PySpark (Madhuri)**

**artist_popularity_skinny**

In [12]:
from pyspark.sql import functions as F

artist_popularity_df = (df.groupBy("artist_name")
                        .agg(F.max("popularity").alias("popularity"))
                        .orderBy(F.desc("popularity"), "artist_name"))

print("artist_popularity_skinny (PySpark DataFrame):")
artist_popularity_df.show(20, truncate=False)

df.createOrReplaceTempView("songs_clean")

# --- Spark SQL version ---
artist_popularity_sql = spark.sql("""
    SELECT
        artist_name,
        MAX(popularity) AS popularity
    FROM songs_clean
    GROUP BY artist_name
    ORDER BY popularity DESC, artist_name
""")

print("artist_popularity_skinny (Spark SQL):")
artist_popularity_sql.show(20, truncate=False)

(artist_popularity_sql
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str("artist_popularity_skinny")))

artist_popularity_skinny (PySpark DataFrame):
+-------------------+----------+
|artist_name        |popularity|
+-------------------+----------+
|Ariana Grande      |100       |
|Post Malone        |99        |
|Daddy Yankee       |98        |
|Ava Max            |97        |
|Halsey             |97        |
|Marshmello         |97        |
|Pedro Capó         |97        |
|Sam Smith          |97        |
|Anuel Aa           |96        |
|DJ Snake           |96        |
|J. Cole            |96        |
|Bad Bunny          |95        |
|Lady Gaga          |95        |
|Meek Mill          |95        |
|Ozuna              |95        |
|Panic! At The Disco|95        |
|Paulo Londra       |95        |
|Calvin Harris      |94        |
|Khalid             |94        |
|Travis Scott       |94        |
+-------------------+----------+
only showing top 20 rows

artist_popularity_skinny (Spark SQL):
+-------------------+----------+
|artist_name        |popularity|
+-------------------+----------+

**freq_artist**

In [13]:
freq_artist_df = (df.groupBy("artist_name")
                  .agg(F.count("*").alias("freq"))
                  .orderBy(F.desc("freq"), "artist_name"))

print("freq_artist (PySpark DataFrame):")
freq_artist_df.show(20, truncate=False)

# --- Spark SQL version -----
freq_artist_sql = spark.sql("""
    SELECT
        artist_name,
        COUNT(*) AS freq
    FROM songs_clean
    GROUP BY artist_name
    ORDER BY freq DESC, artist_name
""")

print("freq_artist (Spark SQL):")
freq_artist_sql.show(20, truncate=False)

(freq_artist_sql
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str("freq_artist")))

freq_artist (PySpark DataFrame):
+------------------------+----+
|artist_name             |freq|
+------------------------+----+
|Giuseppe Verdi          |1141|
|Kimbo Children's Music  |971 |
|Giacomo Puccini         |930 |
|Wolfgang Amadeus Mozart |788 |
|Nobuo Uematsu           |773 |
|Juice Music             |684 |
|Richard Wagner          |681 |
|Randy Newman            |662 |
|Georges Bizet           |627 |
|Johann Sebastian Bach   |611 |
|Ludwig van Beethoven    |585 |
|Hans Zimmer             |559 |
|Chorus                  |480 |
|Henri Salvador          |474 |
|John Williams           |449 |
|Frédéric Chopin         |434 |
|Gioachino Rossini       |428 |
|Bob Marley & The Wailers|380 |
|Dorothée                |378 |
|Children Songs Company  |371 |
+------------------------+----+
only showing top 20 rows

freq_artist (Spark SQL):
+------------------------+----+
|artist_name             |freq|
+------------------------+----+
|Giuseppe Verdi          |1141|
|Kimbo Children's Mu

OPTIONAL: If you want to save the results to your drive:

In [14]:
# --- Optional: saving output next to your cleaned file ---
import os
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# Note: this requires you to have the spotify_cleaned.csv file
# somewhere in your drive already
clean_path = Path("/content/drive").rglob("spotify_cleaned.csv")
clean_path = next(clean_path)  # first match
OUT_DIR = clean_path.parent / "analysis"
if (os.path.exists(OUT_DIR) == False):
  os.makedirs(OUT_DIR, exist_ok=False)

(artist_popularity_sql
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(OUT_DIR / "artist_popularity_skinny")))

(freq_artist_sql
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(OUT_DIR / "freq_artist")))


Mounted at /content/drive
